# HOW TO USE (Python800 version)
1. In the **Configuration** cell, edit ONLY `REAL_CODENET_PATH` so it points to your extracted **Project_CodeNet_Python800** folder (the one containing the problem folders `p00000`, `p00001`, ...). The notebook searches a few levels deep, so an extra nesting level is fine.
2. Make sure `aigcodeset_clean.parquet` is in the same folder as this notebook.
3. Click **Run All**. It must end with **ALL CHECKS PASSED**.
4. Then commit `codenet_subset.parquet`, `problems.parquet`, `codenet_log.csv` in GitHub Desktop and push.

**What changed vs. the full-CodeNet version:** no `metadata/`, `data/` or `problem_descriptions/` folders are needed. Python800 has no submission status, so every row is labelled `Accepted`. Python 2 files are removed by requiring that the code parses with `ast.parse`.

In [1]:
%pip install -q pandas numpy pyarrow beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# WhoWroteIt? - Phase 1 - Person 2: CodeNet subsetting and extraction (Python800)

**Outputs:** `codenet_subset.parquet`, `problems.parquet`, `codenet_log.csv`

**What this notebook does (in order):**
1. Load the clean AIGCodeSet (we need its IDs and code to remove overlap)
2. Find the Python800 problem folders; split into Pool A (AIGCodeSet problems) and Pool B (others)
3. List candidate files per problem (oversampled), drop AIGCodeSet submission IDs
4. Read the code files and apply the exact AIGCodeSet text cleaning
5. Filter by size, drop comment-only files, require valid Python 3 syntax
6. Remove duplicates inside CodeNet
7. Remove overlap with AIGCodeSet (hash layers)
8. Cap per problem, assign `role`, trim to target size
9. Build `problems.parquet` (uses HTML descriptions only if a `problem_descriptions` folder exists)
10. Final checks + write files

In [2]:
import os, re, ast, math, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

## 0. Configuration
**Only change `REAL_CODENET_PATH`.** Everything else is already tuned to match AIGCodeSet's data.

In [3]:
# ---------------------------------------------------------------
# The ONLY line you must edit is REAL_CODENET_PATH.
# ---------------------------------------------------------------
REAL_CODENET_PATH = r"C:\datamining\Project_CodeNet_Python800"   # <-- folder with p00000, p00001, ... (may be nested a few levels)

AIG_PATH = Path("aigcodeset_clean.parquet")   # output of the AIGCodeSet cleaning step (in the repo folder)
OUT_DIR = Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42                      # every random step uses this -> reproducible
TARGET_TRAIN_EXTRA = 20000     # total 'train_extra' human rows (Pool A + Pool B)
MIN_CAP_PER_PROBLEM = 15       # at least 15 rows per problem (same as AIGCodeSET); raised automatically to reach the target
RAG_EXTRA_PER_PROBLEM = 5      # extra rows per problem for the RAG corpus (0 = disable)
OVERSAMPLE = 3                 # read 3x more candidates than needed, because filters will drop some

MIN_CHARS, MAX_CHARS = 50, 5000    # AIGCodeSet's AI max is 5,170
MIN_LINES, MAX_LINES = 3, 200      # AIGCodeSet's AI max is 222

PROBLEM_RE = re.compile(r"^p\d{5}$")

def find_problem_root(base, max_depth=3):
    """Return the folder that directly contains many pXXXXX problem folders (searches a few levels deep)."""
    level = [Path(base)]
    for _ in range(max_depth + 1):
        for d in level:
            if d.is_dir():
                n = sum(1 for x in d.iterdir() if x.is_dir() and PROBLEM_RE.match(x.name))
                if n >= 10:
                    return d
        level = [c for d in level if d.is_dir() for c in d.iterdir() if c.is_dir()]
    return None

CODENET_ROOT = find_problem_root(REAL_CODENET_PATH)
if CODENET_ROOT is None:
    raise FileNotFoundError(
        f"No problem folders (p00000, p00001, ...) found under {REAL_CODENET_PATH}.\n"
        "Point REAL_CODENET_PATH at the extracted Project_CodeNet_Python800 folder.")
if not AIG_PATH.exists():
    raise FileNotFoundError(f"{AIG_PATH} not found. Run this notebook from the repo folder that contains aigcodeset_clean.parquet.")

problem_dirs = sorted(d for d in CODENET_ROOT.iterdir() if d.is_dir() and PROBLEM_RE.match(d.name))

# Optional: HTML problem descriptions (NOT part of Python800). Used only if the folder exists.
DESC_DIR = next((p for p in (CODENET_ROOT / "problem_descriptions",
                             CODENET_ROOT.parent / "problem_descriptions") if p.is_dir()), None)

print("Paths OK ->", CODENET_ROOT)
print("Problem folders:", len(problem_dirs), "| problem_descriptions folder:", DESC_DIR or "not found (descriptions will be empty)")

rng = np.random.default_rng(SEED)
log = {"problem_folders": len(problem_dirs)}   # row counts after every step -> codenet_log.csv

Paths OK -> C:\datamining\Project_CodeNet_Python800
Problem folders: 800 | problem_descriptions folder: not found (descriptions will be empty)


## 1. Shared helper functions
`clean_text` is **exactly the AIGCodeSet cleaning**. If we hashed code without it, identical code would
get different hashes and the overlap check would silently miss it.

In [4]:
def clean_text(s: str) -> str:
    """the AIGCodeSet cleaning: line endings, BOM, NUL."""
    return (str(s).replace("\r\n", "\n").replace("\r", "\n")
                  .replace("\ufeff", "").replace("\x00", ""))

def loose_text(s: str) -> str:
    """Looser version: strip trailing spaces on each line + strip the whole file.
    Catches copies that differ only in whitespace."""
    return "\n".join(line.rstrip() for line in s.split("\n")).strip()

def md5(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def has_real_code(s: str) -> bool:
    """False if the file is only blank lines / comments."""
    return any(l.strip() and not l.strip().startswith("#") for l in s.split("\n"))

def parses_ok(s: str) -> bool:
    """True if the code is valid Python 3 syntax (also removes Python 2 files)."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ast.parse(s)
        return True
    except (SyntaxError, ValueError, RecursionError, MemoryError):
        return False

## 2. Load the clean AIGCodeSet
We need three things from it, to remove overlap later:
- her **problem IDs** (= Pool A)
- her **submission IDs** - *both* human rows *and* AI rows (AI rows store the ID of the buggy
  submission the LLM fixed; the from-scratch ones say `"unrelated"`, which we ignore)
- her **code hashes** (raw and loose)

In [5]:
aig = pd.read_parquet(AIG_PATH)
print(aig.shape, "| problems:", aig.problem_id.nunique())

aig_problems = set(aig.problem_id)
aig_sub_ids  = set(aig.submission_id) - {"unrelated"}
aig_hash_raw   = {md5(clean_text(c)) for c in aig.code_raw}
aig_hash_loose = {md5(loose_text(clean_text(c))) for c in aig.code_raw}

log["aig_rows"] = len(aig)
log["aig_problems"] = len(aig_problems)
log["aig_submission_ids"] = len(aig_sub_ids)

(7424, 12) | problems: 317


## 3. Pick problems: Pool A and Pool B
- **Pool A** = AIGCodeSet problems that exist in the Python800 folder. They have BOTH human and AI samples, so val/test must come only from here.
- **Pool B** = all other Python800 problems -> extra variety for training and RAG only.

Python800 has only ~800 problems, so instead of the old "15 per problem, add more problems" logic, the per-problem cap is
raised automatically so that `TARGET_TRAIN_EXTRA` can be reached.

In [6]:
all_problems = [d.name for d in problem_dirs]
pool_a = sorted(set(all_problems) & aig_problems)
pool_b = sorted(set(all_problems) - aig_problems)

CAP_PER_PROBLEM = max(MIN_CAP_PER_PROBLEM, math.ceil(TARGET_TRAIN_EXTRA / len(all_problems)))

print(f"Pool A: {len(pool_a)} of {len(aig_problems)} AIGCodeSet problems found in this folder")
print(f"Pool B: {len(pool_b)} problems")
print(f"Cap per problem: {CAP_PER_PROBLEM} train_extra + {RAG_EXTRA_PER_PROBLEM} rag_only")
if len(pool_a) == 0:
    print("WARNING: none of the AIGCodeSet problems are in this folder. Pool A is empty -> no val/test pool. Check the problem IDs.")
elif len(pool_a) < len(aig_problems):
    print(f"NOTE: {len(aig_problems) - len(pool_a)} AIGCodeSet problems are not in Python800 (they will be missing from CodeNet).")

log["pool_a_problems"] = len(pool_a)
log["pool_b_problems"] = len(pool_b)
log["cap_per_problem"] = CAP_PER_PROBLEM

Pool A: 317 of 317 AIGCodeSet problems found in this folder
Pool B: 483 problems
Cap per problem: 25 train_extra + 5 rag_only


## 4. Choose which files to read (oversampled, per problem)
Python800 has no metadata, so there is nothing to filter yet: list the `.py` files of each problem, drop every submission ID
that exists in AIGCodeSet (**overlap layer 1**, the file name is the submission ID), shuffle, and keep the first
`(cap + rag) * OVERSAMPLE` files of each problem.

In [7]:
need_per_problem = CAP_PER_PROBLEM + RAG_EXTRA_PER_PROBLEM
rows = []
n_files = n_l1 = 0
for i, pid in enumerate(pool_a + pool_b):
    pdir = CODENET_ROOT / pid
    files = sorted(pdir.glob("*.py")) or sorted(pdir.rglob("*.py"))
    n_files += len(files)
    kept = [f for f in files if f.stem not in aig_sub_ids]                  # OVERLAP LAYER 1
    n_l1 += len(files) - len(kept)
    for j in rng.permutation(len(kept))[: need_per_problem * OVERSAMPLE]:
        rows.append({"problem_id": pid, "submission_id": kept[j].stem, "path": kept[j]})
    if (i + 1) % 200 == 0:
        print(f"  listed {i+1}/{len(pool_a) + len(pool_b)} problems")

cands = pd.DataFrame(rows)
cands["in_aigcodeset_problem"] = cands["problem_id"].isin(aig_problems)
cands["status"] = "Accepted"    # Python800 has no status info; its files are accepted solutions
log["all_py_files"] = n_files
log["overlap_L1_submission_id_removed"] = n_l1
log["candidates_to_read"] = len(cands)
print("py files:", n_files, "| removed by submission_id:", n_l1, "| files to read:", len(cands))

  listed 200/800 problems
  listed 400/800 problems
  listed 600/800 problems
  listed 800/800 problems
py files: 240000 | removed by submission_id: 421 | files to read: 72000


## 5. Read the code files + apply the AIGCodeSet cleaning

In [8]:
def read_code(p):
    try:
        return p.read_bytes().decode("utf-8", errors="replace")
    except OSError:
        return None

cands["code_raw"] = [read_code(p) for p in cands["path"]]
n0 = len(cands)
cands = cands.dropna(subset=["code_raw"])
log["unreadable_files_removed"] = n0 - len(cands)

cands["code_raw"] = cands["code_raw"].map(clean_text)
n0 = len(cands)
cands = cands[~cands["code_raw"].str.contains("\ufffd", regex=False)]     # broken encoding
log["bad_encoding_removed"] = n0 - len(cands)
log["after_read_and_clean"] = len(cands)

## 6. Size filters, comment-only files, syntax check
`n_chars` / `n_lines` use **the same formulas as AIGCodeSet**. Without metadata we cannot tell Python 2 from Python 3,
so we require `ast.parse` to succeed. That removes Python 2 files (e.g. `print x`) and broken files.

In [9]:
cands["n_chars"] = cands["code_raw"].str.len()
cands["n_lines"] = cands["code_raw"].str.count("\n") + 1

n0 = len(cands)
cands = cands[cands["n_chars"].between(MIN_CHARS, MAX_CHARS) & cands["n_lines"].between(MIN_LINES, MAX_LINES)]
log["size_filter_removed"] = n0 - len(cands)

n0 = len(cands)
cands = cands[cands["code_raw"].map(has_real_code)]
log["comment_or_blank_only_removed"] = n0 - len(cands)

cands["parses_ok"] = cands["code_raw"].map(parses_ok)
n0 = len(cands)
cands = cands[cands["parses_ok"]]
log["unparsable_removed"] = n0 - len(cands)
log["after_size_and_syntax"] = len(cands)
print(log["after_size_and_syntax"], "rows kept | unparsable (mostly Python 2) removed:", log["unparsable_removed"])

67437 rows kept | unparsable (mostly Python 2) removed: 1221


## 7. Dedup inside CodeNet, then overlap layers 2 and 3 against AIGCodeSet
- hash 1 = raw (after the AIGCodeSet cleaning) - exact match
- hash 2 = loose (trailing spaces stripped) - near-exact match

In [10]:
cands["code_hash_raw"]   = cands["code_raw"].map(md5)
cands["_hash_loose"]     = cands["code_raw"].map(lambda s: md5(loose_text(s)))

n0 = len(cands)
cands = cands.drop_duplicates("code_hash_raw").drop_duplicates("_hash_loose")
log["internal_duplicates_removed"] = n0 - len(cands)

n0 = len(cands)
cands = cands[~cands["code_hash_raw"].isin(aig_hash_raw)]                # OVERLAP LAYER 2
log["overlap_L2_raw_hash_removed"] = n0 - len(cands)

n0 = len(cands)
cands = cands[~cands["_hash_loose"].isin(aig_hash_loose)]                # OVERLAP LAYER 3
log["overlap_L3_loose_hash_removed"] = n0 - len(cands)
log["after_overlap_removal"] = len(cands)

## 8. Cap per problem, assign `role`, trim to target size
- first `CAP_PER_PROBLEM` rows of each problem -> `train_extra`
- next `RAG_EXTRA_PER_PROBLEM` rows -> `rag_only` (Yassin must NOT train on these)
- If `train_extra` is above `TARGET_TRAIN_EXTRA`, the excess is removed from **Pool B** first (Pool A is kept).

In [11]:
cands = cands.sample(frac=1, random_state=SEED).reset_index(drop=True)
cands["_rank"] = cands.groupby("problem_id").cumcount()
cands["role"] = np.where(cands["_rank"] < CAP_PER_PROBLEM, "train_extra",
                 np.where(cands["_rank"] < CAP_PER_PROBLEM + RAG_EXTRA_PER_PROBLEM, "rag_only", "drop"))
final = cands[cands["role"] != "drop"].copy()

excess = int((final["role"] == "train_extra").sum()) - TARGET_TRAIN_EXTRA
if excess > 0:
    b_idx = final[(~final["in_aigcodeset_problem"]) & (final["role"] == "train_extra")].index.to_numpy()
    drop_idx = rng.choice(b_idx, size=min(excess, len(b_idx)), replace=False)
    final = final.drop(index=drop_idx)
    log["trimmed_to_target"] = len(drop_idx)

n_train = int((final["role"] == "train_extra").sum())
log["target_train_extra"] = TARGET_TRAIN_EXTRA
log["train_extra_shortfall"] = max(TARGET_TRAIN_EXTRA - n_train, 0)
if n_train < TARGET_TRAIN_EXTRA:
    print(f"NOTE: only {n_train} train_extra rows available (target {TARGET_TRAIN_EXTRA}). Python800 is small; this is expected.")

final = final.reset_index(drop=True)
log["final_pool_a_problems"] = final[final.in_aigcodeset_problem].problem_id.nunique()
log["final_pool_b_problems"] = final[~final.in_aigcodeset_problem].problem_id.nunique()
print(final.groupby(["in_aigcodeset_problem", "role"]).size())

NOTE: only 19998 train_extra rows available (target 20000). Python800 is small; this is expected.
in_aigcodeset_problem  role       
False                  rag_only        2410
                       train_extra    12073
True                   rag_only        1585
                       train_extra     7925
dtype: int64


## 9. Build the final schema (AIGCodeSet's column names + our additions)

In [12]:
final["sample_id"]   = "cn_" + final["submission_id"]
final["source"]      = "codenet"
final["language"]    = "python"
final["label"]       = 0
final["llm_name"]    = "human"
final["scenario"]    = "human"
final["rag_eligible"] = True          # every Python800 row is an accepted solution

cols = ["sample_id", "source", "submission_id", "problem_id", "language", "code_raw",
        "label", "llm_name", "scenario", "status", "n_lines", "n_chars",
        "parses_ok", "in_aigcodeset_problem", "rag_eligible", "role", "code_hash_raw"]
codenet_subset = final[cols].sort_values(["problem_id", "sample_id"]).reset_index(drop=True)

## 10. Problem descriptions -> `problems.parquet`
Python800 does **not** include problem descriptions. If a `problem_descriptions` folder exists next to the problem folders,
HTML is converted to text as before. Otherwise `problems.parquet` is still written (so the next steps do not break), but
`description` is empty and `has_description` is `False`.

In [13]:
INLINE_TAGS = ["var", "span", "strong", "b", "em", "i", "code", "a", "u", "font"]

def html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    for t in soup.find_all("sup"):
        t.replace_with("^" + t.get_text())
    for t in soup.find_all("sub"):
        t.replace_with("_" + t.get_text())
    for t in soup.find_all(INLINE_TAGS):
        t.unwrap()                      # keep the text, remove the tag (no line break)
    soup.smooth()
    text = soup.get_text("\n")
    lines = [re.sub(r"[ \t]+", " ", l).strip() for l in text.split("\n")]
    text = "\n".join(lines)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def get_title(html: str, fallback: str) -> str:
    h1 = BeautifulSoup(html, "lxml").find("h1")
    return h1.get_text(strip=True) if h1 else fallback

need_problems = sorted(set(codenet_subset.problem_id) | aig_problems)
rows, missing = [], []
for pid in need_problems:
    p = DESC_DIR / f"{pid}.html" if DESC_DIR else None
    if p is not None and p.exists():
        html = p.read_text(encoding="utf-8", errors="replace")
        desc, title = html_to_text(html), get_title(html, pid)
    else:
        missing.append(pid)
        desc, title = "", pid
    rows.append({
        "problem_id": pid,
        "title": title,
        "description": desc,
        "n_chars": len(desc),
        "has_description": bool(desc),
        "in_aigcodeset_problem": pid in aig_problems,
    })
problems = pd.DataFrame(rows)
log["problems_needed"] = len(need_problems)
log["problems_with_description"] = int(problems.has_description.sum())
log["problems_missing_description"] = len(missing)
print("descriptions:", log["problems_with_description"], "of", len(need_problems), "| missing:", len(missing))
if DESC_DIR is None:
    print("NOTE: no problem_descriptions folder -> problems.parquet has titles = problem ids and empty descriptions.")

descriptions: 0 of 800 | missing: 800
NOTE: no problem_descriptions folder -> problems.parquet has titles = problem ids and empty descriptions.


## 11. Final checks (these `assert`s must all pass before you push)

In [14]:
cs = codenet_subset
assert len(cs) > 0, "empty subset"
assert cs["sample_id"].is_unique
assert (cs["label"] == 0).all() and (cs["source"] == "codenet").all()
assert (cs["status"] == "Accepted").all()
assert cs["code_raw"].notna().all() and (cs["code_raw"].str.strip() != "").all()
assert cs["n_chars"].between(MIN_CHARS, MAX_CHARS).all()
assert cs["parses_ok"].all(), "every row must parse as Python 3"

# zero overlap with AIGCodeSet, all 3 layers
assert not cs["submission_id"].isin(aig_sub_ids).any(),                      "overlap L1 (submission_id)"
assert not cs["code_hash_raw"].isin(aig_hash_raw).any(),                     "overlap L2 (raw hash)"
assert not cs["code_raw"].map(lambda s: md5(loose_text(s))).isin(aig_hash_loose).any(), "overlap L3 (loose hash)"

# problems table covers the subset
assert set(cs["problem_id"]) <= set(problems["problem_id"]), "subset problem missing in problems table"
if missing:
    print(f"WARNING: {len(missing)} problems without description (expected if there is no problem_descriptions folder)")

# rag_only rows must be rag eligible
assert cs.loc[cs.role == "rag_only", "rag_eligible"].all()

print("ALL CHECKS PASSED")
print("\nlength comparison (AIGCodeSet human vs CodeNet extra):")
print(pd.DataFrame({"aig_human": aig[aig.label == 0].n_chars.describe(),
                    "codenet_extra": cs.n_chars.describe()}).round(0))
print("\nrole counts:\n", cs.role.value_counts())

ALL CHECKS PASSED

length comparison (AIGCodeSet human vs CodeNet extra):
       aig_human  codenet_extra
count     4750.0        23993.0
mean       464.0          374.0
std        648.0          385.0
min          1.0           50.0
25%        151.0          143.0
50%        302.0          247.0
75%        574.0          466.0
max      19752.0         4969.0

role counts:
 role
train_extra    19998
rag_only        3995
Name: count, dtype: int64


## 12. Save

In [15]:
log["final_rows"] = len(cs)
for role, n in cs.groupby("role").size().items():
    log[f"final_{role}"] = int(n)
log["final_pool_a_rows"] = int(cs.in_aigcodeset_problem.sum())
log["final_pool_b_rows"] = int((~cs.in_aigcodeset_problem).sum())

cs.to_parquet(OUT_DIR / "codenet_subset.parquet", index=False)
problems.to_parquet(OUT_DIR / "problems.parquet", index=False)
pd.Series(log).to_csv(OUT_DIR / "codenet_log.csv", header=["count"])
print(pd.Series(log).to_string())

problem_folders                        800
aig_rows                              7424
aig_problems                           317
aig_submission_ids                    4751
pool_a_problems                        317
pool_b_problems                        483
cap_per_problem                         25
all_py_files                        240000
overlap_L1_submission_id_removed       421
candidates_to_read                   72000
unreadable_files_removed                 0
bad_encoding_removed                     0
after_read_and_clean                 72000
size_filter_removed                   3342
comment_or_blank_only_removed            0
unparsable_removed                    1221
after_size_and_syntax                67437
internal_duplicates_removed              0
overlap_L2_raw_hash_removed             14
overlap_L3_loose_hash_removed            5
after_overlap_removal                67418
target_train_extra                   20000
train_extra_shortfall                    2
final_pool_

In [ ]:
import os
from IPython.display import display

a = pd.read_parquet(OUT_DIR / "codenet_subset.parquet")
b = pd.read_parquet(OUT_DIR / "problems.parquet")

for f in ["codenet_subset.parquet", "problems.parquet", "codenet_log.csv"]:
    print(f"{f:26s}{os.path.getsize(OUT_DIR / f) / 1024:10.1f} KB")

print("\ncodenet_subset.parquet ->", a.shape[0], "rows x", a.shape[1], "columns")
print("roles:", a.role.value_counts().to_dict(), "| problems:", a.problem_id.nunique())
display(a.drop(columns=["code_raw", "code_hash_raw"]).head(5))

print("\nproblems.parquet ->", b.shape[0], "rows x", b.shape[1], "columns")
display(b.assign(description=b["description"].str[:60]).head(5))